In [1]:
import os
import zipfile
from datetime import datetime

import arcpy
from arcgis.gis import GIS

In [2]:
def utworz_nazwe_backupu():
    data = datetime.now().strftime("%Y-%m-%d")
    return f"Backup_AGOL_{data}"

In [3]:
def utworz_geobaze(folder_docelowy, nazwa_gdb):
    sciezka_gdb = os.path.join(folder_docelowy, nazwa_gdb)

    if not arcpy.Exists(sciezka_gdb):
        arcpy.CreateFileGDB_management(folder_docelowy, nazwa_gdb)

    return sciezka_gdb

In [4]:
def pobierz_hosted_layers(gis):
    user = gis.users.me
    items = user.items(max_items=1000)

    hosted_layers = []

    for item in items:
        if item.type == "Feature Service" and "Hosted Service" in item.typeKeywords:
            hosted_layers.append(item)

    # hosted_layers = gis.content.search(query="title: 'geol_punkty_Szczawa_2024_ES' AND type: 'Feature Service'")
    hosted_layers = hosted_layers[:2]
    return hosted_layers
    # pobieram tylko dwie bo to dużo roboty dla większej ilości

In [5]:
import shutil

def eksportuj_warstwy_do_gdb(hosted_layers, sciezka_gdb):
    folder_roboczy = os.path.dirname(sciezka_gdb)
    
    for layer in hosted_layers:
        try:
            print(f"--- Przetwarzanie: {layer.title} ---")

            # 1. Eksport do tymczasowego FGDB w chmurze
            export_item = layer.export(
                title=f"temp_export_{layer.id}",
                export_format="File Geodatabase"
            )

            # 2. Pobranie ZIPa na dysk
            zip_path = export_item.download(save_path=folder_roboczy)
            
            # 3. Rozpakowanie ZIPa do folderu tymczasowego
            temp_extract_dir = os.path.join(folder_roboczy, f"temp_{layer.id}")
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(temp_extract_dir)

            # 4. Znalezienie bazy .gdb wewnątrz wypakowanych plików
            for root, dirs, _ in os.walk(temp_extract_dir):
                for d in dirs:
                    if d.endswith(".gdb"):
                        gdb_tymczasowa = os.path.join(root, d)
                        
                        # Ustawienie środowiska na pobraną bazę
                        arcpy.env.workspace = gdb_tymczasowa
                        fcs = arcpy.ListFeatureClasses()
                        
                        for fc in fcs:
                            # Tworzymy bezpieczną nazwę dla nowej klasy obiektów
                            # Dodajemy tytuł warstwy, by uniknąć duplikatów nazw (np. 'Layer_0')
                            nowa_nazwa = arcpy.ValidateTableName(f"{layer.title}_{fc}", sciezka_gdb)
                            
                            # KOPIOWANIE: z tymczasowej bazy do Twojej bazy głównej
                            arcpy.FeatureClassToFeatureClass_conversion(
                                fc, 
                                sciezka_gdb, 
                                nowa_nazwa
                            )
                            print(f"  ✔ Dodano do bazy: {nowa_nazwa}")

            # 5. Sprzątanie
            export_item.delete()             # Usuwa z AGOL
            os.remove(zip_path)               # Usuwa ZIP z dysku
            shutil.rmtree(temp_extract_dir)   # Usuwa rozpakowany folder
            print(f"  🗑 Wyczyszczono pliki tymczasowe")

        except Exception as e:
            print(f"  ❌ Błąd podczas importu {layer.title}: {e}")

In [6]:
def zipuj_geobaze(sciezka_gdb, sciezka_zip):
    with zipfile.ZipFile(sciezka_zip, "w", zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(sciezka_gdb):
            for file in files:
                if file.endswith('.lock'):
                    continue
                
                full_path = os.path.join(root, file)
                rel_path = os.path.relpath(
                    full_path,
                    os.path.dirname(sciezka_gdb)
                )
                zipf.write(full_path, rel_path)

In [7]:
gis = GIS("home")

print("Zalogowano jako:", gis.users.me.username)

Zalogowano jako: eskwarczek


In [8]:
folder_backupu = r"C:\Users\edyta\Documents\Studia\8 semestr\Systemy geoinformatyczne\cwl7"

nazwa_backupu = utworz_nazwe_backupu()
nazwa_gdb = f"{nazwa_backupu}.gdb"

sciezka_gdb = utworz_geobaze(folder_backupu, nazwa_gdb)
sciezka_zip = os.path.join(folder_backupu, f"{nazwa_backupu}.zip")

In [9]:
hosted_layers = pobierz_hosted_layers(gis)

print(f"Znaleziono {len(hosted_layers)} hosted feature layers")

Znaleziono 2 hosted feature layers


In [10]:
eksportuj_warstwy_do_gdb(hosted_layers, sciezka_gdb)

--- Przetwarzanie: geol_punkty_Szczawa_2024_ES ---
  ✔ Dodano do bazy: geol_punkty_Szczawa_2024_ES_geol_punkty
  🗑 Wyczyszczono pliki tymczasowe
--- Przetwarzanie: tereny_badan_2024_ES ---
  ✔ Dodano do bazy: tereny_badan_2024_ES_tereny_badan
  🗑 Wyczyszczono pliki tymczasowe


In [11]:
zipuj_geobaze(sciezka_gdb, sciezka_zip)

print("Backup zapisany w:")
print(sciezka_zip)

Backup zapisany w:
C:\Users\edyta\Documents\Studia\8 semestr\Systemy geoinformatyczne\cwl7\Backup_AGOL_2026-05-14.zip
